In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from core.datasetclass import BenchmarkDataset

/dolfinx-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@jax.vmap
def transformation_jacobian(coords_elem) :
    x1, y1 = coords_elem[0]
    x2, y2 = coords_elem[1]
    x3, y3 = coords_elem[2]

    # Jacobian of shape function derivatives
    J = jnp.array([
        [x2 - x1, y2 - y1],
        [x3 - x1, y3 - y1]
    ])
    return J

In [3]:
@jax.vmap
def deformation_gradient_element(coords_elem, disp_elem):
    x1, y1 = coords_elem[0]
    x2, y2 = coords_elem[1]
    x3, y3 = coords_elem[2]

    # Jacobian of shape function derivatives
    J = jnp.array([
        [x2 - x1, y2 - y1],
        [x3 - x1, y3 - y1]
    ])

    # Area factor
    detJ = jnp.linalg.det(J)

    # Shape function derivatives in reference space
    dN_ref = jnp.array([
        [-1., -1.],
        [ 1.,  0.],
        [ 0.,  1.]
    ])

    # Convert to physical derivatives: dN/dx = inv(J)^T * dN_ref
    dNdx = jnp.transpose(jnp.linalg.solve(J, dN_ref.T))

    # Gradient of displacement
    gradu = disp_elem.T @ dNdx  # 2x3 @ 3x2 = 2x2

    # Deformation gradient
    F = jnp.eye(2) + gradu
    return F, dNdx

In [4]:
dataset = BenchmarkDataset("dataset/benchmarks", "noise=low", "NeoHookean")

In [5]:
data = dataset[10]

/home/mmdiscovery/shared/core/datasetclass.py:205: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  output_nodes.ux[output_nodes.bcx!=0] = output_nodes.ux_orig[output_nodes.bcx!=0]
/home/mmdiscovery/shared/core/datasetclass.py:206: FutureWarnin

In [59]:
cells = data["cells"]
reaction_forces = jnp.array(data["reaction_forces"].to_numpy())

# Weak Form

$$f_i^a = \int P_{ij} \nabla_j N^a dV$$ and $$\sum^{N_a}_{a=1} v_i^a f_i^a = 0$$

In [60]:
from core.utils import C_func, I1_func, I3_func

In [61]:
def NH_psi(F, mu = 1.0, kappa = 3.0) :
    if F.shape[-2:] == (2, 2) :
        F = jnp.array([[F[0, 0], F[0, 1], 0.], 
                       [F[1, 0], F[1, 1], 0.],
                       [0., 0., 1.]])
    C = C_func(F)
    I1 = I1_func(C)
    J = jnp.sqrt(I3_func(C))
    psi = mu/2 * (J**(-2/3) * I1 - 3) + kappa/2 * (J - 1)**2 
    return psi

P_nh = jax.vmap(jax.grad(NH_psi, 0))

In [83]:
def NH_psi(F, params):
    mu, kappa = params
    if F.shape[-2:] == (2, 2):
        F = jnp.array([[F[0, 0], F[0, 1], 0.], 
                       [F[1, 0], F[1, 1], 0.],
                       [0.,      0.,     1. ]])

    C = C_func(F)
    I1 = I1_func(C)
    J  = jnp.sqrt(I3_func(C))

    psi = mu/2 * (J**(-2/3) * I1 - 3) + kappa/2 * (J - 1)**2
    return psi


P_nh = jax.vmap(jax.grad(NH_psi, argnums=0), in_axes=(0, None))
def physical_loss(params, coords, cells, u, reaction_forces, n_nodes, bc):
    mu, kappa = params

    # compute deformation gradient + shape function gradients
    F, dNdx = deformation_gradient_element(coords, u)
    dA = jnp.linalg.det(transformation_jacobian(coords)) / 2

    # compute 1st Piola stress
    piola = P_nh(F, params)    # shape (C, 2, 2)

    # element forces: (C, 2, 3)
    f_int_cell  = jnp.einsum("cij, cnj -> cin", piola, dNdx) * dA[:, None, None]
    f_int_cell  = jnp.swapaxes(f_int_cell, 1, 2)              # (C, 3, 2)

    # assemble into global vector

    f_int_nodes = jnp.zeros((n_nodes, 2)).at[cells].add(f_int_cell)

    # Dirichlet residual (only on free DOFs)
    blm_loss = jnp.sum(f_int_nodes[bc == 0] ** 2)

    # reaction matching loss
    reaction_ids = jnp.arange(1, 5)  # (4,)

    # reaction_mask[r, n, d] = 1 if bc[n,d] == r+1
    reaction_mask = (bc[None, :, :] == reaction_ids[:, None, None]).astype(f_int_nodes.dtype)
    # shape = (4, n_nodes, 2)

    # force for each reaction_id: sum over nodes & dofs
    f_react = jnp.sum(reaction_mask * f_int_nodes[None, :, :], axis=(1, 2))  # (4,)

    # target reactions must be JAX array
    r = jnp.asarray(reaction_forces).reshape(-1)  # (4,)

    reaction_loss = jnp.sum((f_react - r)**2)

    return blm_loss + reaction_loss


# def physical_loss(coords, cells, u, reaction_forces, P_func) :
#     F, dNdx = deformation_gradient_element(coords, u)
#     dA = jnp.linalg.det(transformation_jacobian(coords))/2
#     piola = P_func(F)
#     f_int_cell_ = jnp.einsum("cij, cnj -> cin", piola, dNdx) * dA[:, None, None]
#     f_int_cell_perm = jnp.swapaxes(f_int_cell_, 1, 2)   # (C, 3, 2)
#     n_nodes = cells.max() + 1
#     f_int_nodes = jnp.zeros((n_nodes, 2))
#     f_int_nodes = f_int_nodes.at[cells].add(f_int_cell_perm)
#     blm_loss = jnp.sum(f_int_nodes[data["bc"] == 0]**2)
#     reaction_loss = 0.0
#     for i in range(1, 4 + 1) :
#         reaction_loss += (reaction_forces[i - 1] - f_int_nodes[data["bc"] == i].sum())**2
#     return blm_loss + reaction_loss, blm_loss, reaction_loss



In [84]:
(jnp.max(cells) + 1)

Array(1441, dtype=int64)

In [87]:
import optax

# trainable parameters
params = jnp.array([1.0, 1.0])   # initial guess [mu, kappa]

# choose optimizer
opt = optax.adam(1e-3)
opt_state = opt.init(params)
n_nodes = int(cells.max()) + 1
# JIT the loss and gradients
loss_and_grad = jax.jit(jax.value_and_grad(
    lambda params: physical_loss(params, data["coords_elems"], cells, data["disp_elems"], reaction_forces, n_nodes, data["bc"])
))


In [89]:

for step in range(10000):
    loss, grads = loss_and_grad(params)
    updates, opt_state = opt.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    if step % 50 == 0:
        print(f"step {step:04d}  loss={loss:.6f}  mu={params[0]:.4f}, kappa={params[1]:.4f}")


step 0000  loss=0.072879  mu=1.7622, kappa=1.8285
step 0050  loss=0.067195  mu=1.7859, kappa=1.8605
step 0100  loss=0.062010  mu=1.8082, kappa=1.8917
step 0150  loss=0.057290  mu=1.8288, kappa=1.9220
step 0200  loss=0.053000  mu=1.8480, kappa=1.9515
step 0250  loss=0.049106  mu=1.8656, kappa=1.9801
step 0300  loss=0.045577  mu=1.8816, kappa=2.0079
step 0350  loss=0.042381  mu=1.8960, kappa=2.0349
step 0400  loss=0.039490  mu=1.9089, kappa=2.0611
step 0450  loss=0.036876  mu=1.9201, kappa=2.0865
step 0500  loss=0.034512  mu=1.9298, kappa=2.1111
step 0550  loss=0.032374  mu=1.9380, kappa=2.1349
step 0600  loss=0.030438  mu=1.9445, kappa=2.1580
step 0650  loss=0.028683  mu=1.9495, kappa=2.1804
step 0700  loss=0.027089  mu=1.9530, kappa=2.2022
step 0750  loss=0.025638  mu=1.9550, kappa=2.2232
step 0800  loss=0.024311  mu=1.9556, kappa=2.2436
step 0850  loss=0.023095  mu=1.9547, kappa=2.2633
step 0900  loss=0.021974  mu=1.9524, kappa=2.2825
step 0950  loss=0.020938  mu=1.9488, kappa=2.3011
